# Weather Anomaly Detection — Budapest (2020–2026)
This notebook detects statistically unusual weather events in Budapest 
using 6 years of hourly Open-Meteo data. Anomalies are identified using 
Z-score analysis grouped by Month and Hour.

# Install Relevent Packages

In [1]:
!pip install openmeteo-requests
!pip install requests-cache retry-requests numpy pandas

# Import the Packages

In [2]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Collect Weather Data From 

In [3]:
import openmeteo_requests
import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 47.4979,
    "longitude": 19.0402,
    "start_date": "2020-01-01",
    "end_date": "2026-04-19",
    "hourly": [
        "temperature_2m",        # Variables(0)
        "relative_humidity_2m",  # Variables(1)
        "precipitation",         # Variables(2)
        "wind_speed_10m",        # Variables(3)
        "apparent_temperature",  # Variables(4)
    ],
    "timezone": "Europe/Budapest"
}

responses = openmeteo.weather_api(url, params=params)
response = responses[0]

print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone offset: {response.UtcOffsetSeconds()}s")

# Extract hourly data
hourly = response.Hourly()

hourly_data = {
    "date": pd.date_range(
        start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
        end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
        freq=pd.Timedelta(seconds=hourly.Interval()),
        inclusive="left"
    ),
    # Extract ALL 5 variables in the SAME order as params
    "temperature_2m":       hourly.Variables(0).ValuesAsNumpy(),
    "relative_humidity_2m": hourly.Variables(1).ValuesAsNumpy(),
    "precipitation":        hourly.Variables(2).ValuesAsNumpy(),
    "wind_speed_10m":       hourly.Variables(3).ValuesAsNumpy(),
    "apparent_temperature": hourly.Variables(4).ValuesAsNumpy(),
}

df = pd.DataFrame(data=hourly_data)

# Convert timezone from UTC to Budapest local time
df["date"] = df["date"].dt.tz_convert("Europe/Budapest")



Coordinates: 47.48681640625°N 19.06403923034668°E
Elevation: 113.0 m asl
Timezone offset: 7200s


# Data cleaning

In [4]:
df.head()

,date,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,apparent_temperature
0,2019-12-31 23:00:00+01:00,0.25,87.058907,0.0,18.584509,-4.826773
1,2020-01-01 00:00:00+01:00,0.20,88.342773,0.0,19.602652,-5.004395
2,2020-01-01 01:00:00+01:00,0.75,84.590553,0.0,19.453327,-4.439463
3,2020-01-01 02:00:00+01:00,0.70,85.835953,0.0,18.844202,-4.379635
4,2020-01-01 03:00:00+01:00,0.35,88.032867,0.0,17.819090,-4.580226


### I find it useful to have an easy to access guide of the dataset parameters as table to not get lost in the data.So i am puting a description on how to read the the parameter ()Column names below.

| Column name                   | Meaning                                                                 | Typical unit |
|------------------------------|-------------------------------------------------------------------------|-------------|
| `date`                       | Timestamp or date of the observation (e.g., hour, day, etc.)          | datetime / ISO string |
| `temperature_2m`             | Air temperature measured about 2 meters above the ground              | °C          |
| `relative_humidity_2m`       | Relative humidity of the air at about 2 meters above the ground       | %           |
| `precipitation`              | Total accumulated precipitation (rain, snow, showers) over the period | mm          |
| `wind_speed_10m`             | Wind speed measured about 10 meters above the ground                  | km/h        |
| `apparent_temperature`       | “Feels like” temperature, combining air temperature and humidity/wind | °C          |

# Data-Cleaning Pipeline 
#### 1. lets see what shape is the data
#### 2. check if there are any missing values
#### 3. the dtypes of the data
#### 4. Are there any duplicate rows

In [5]:
df.shape # the data set has 55,224 rows and 6 columns


(55224, 6)

In [6]:
df.isnull().sum() #Perfect!! no missing data

date                    0
temperature_2m          0
relative_humidity_2m    0
precipitation           0
wind_speed_10m          0
apparent_temperature    0
dtype: int64

In [7]:
df.info() # Everything seems to be in order so far

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55224 entries, 0 to 55223
Data columns (total 6 columns):
 #   Column                Non-Null Count  Dtype                          
---  ------                --------------  -----                          
 0   date                  55224 non-null  datetime64[ns, Europe/Budapest]
 1   temperature_2m        55224 non-null  float32                        
 2   relative_humidity_2m  55224 non-null  float32                        
 3   precipitation         55224 non-null  float32                        
 4   wind_speed_10m        55224 non-null  float32                        
 5   apparent_temperature  55224 non-null  float32                        
dtypes: datetime64[ns, Europe/Budapest](1), float32(5)
memory usage: 1.5 MB


In [8]:
df.duplicated().value_counts() # there dosnt seem to be any duplicated values.

False    55224
Name: count, dtype: int64

### Now that we have completed the data cleaning , it's time to move on to the next step.

### Since the aim is to do anomaly detection on this weather data , we will be using the following pipe line

### Workflow of Weather Anomaly Detection 
#### 1. Data Collection: Gathering historical and real-time weather data from sensors, satellites, and stations. ✅
#### 2. Preprocessing: Cleaning the data by filling in missing values and removing noise. ✅
#### 3. Baseline Modeling: Using historical data to define "normal" weather patterns.
#### 4. Anomaly Detection: Applying ML or statistical algorithms to new data to identify deviations from the baseline.
#### 5. Validation: Interpreting results, reducing false positives, and identifying extreme weather events. 

# Theory

The Core Concept that Needs to be Internalized **A baseline question you need to answer: "What is normal for this exact moment in time?"** Not normal overall. Normal right now — this hour, this month, this season.

well in this note book i use z- score analysis.Which is a statistical method used to determine how many standard deviations a specific data point is from the mean of a dataset.so eg- if we look at anomallies in temperature then how much does the std move from the temperature - the mean temp.
but in order to do that we have to answer another important question.

**For our problem how world we selet the time period over which the anomally occurs?**
**Answer:** 

**First->** Using the pandas .dt function lets split the date coulumn into three by = "Month", "Hour" , "day of year"

**Second->** What should I group by? — month only? month + hour? day of year? Think about what "same conditions" means for temperature at 3am in January vs 3pm in July.
now lets do some maths , Boring i know but nessesary.

**A)** If we group by **Day of Year + Hour**, we get 365 × 24 = **8,784 groups**
— Very specific, but with ~6 years of data that gives us only ~6 data points per group. Too few for a reliable standard deviation.Think about that. If you want to know the "normal" temperature at 14:00 on April 19th, you only have 6 April 19th readings across 6 years. One unusually hot year will completely distort your std. Your anomaly detector will be unreliable and noisy.

**B)** If we group by **Month + Hour**, we get 12 × 24 = **288 groups**
— Each group has ~55,224 ÷ 288 = **~192 data points** across 6 years. Reliable  — **this is what we chose.**

**C)** If we group by **Month + Day**, we get 12 × 31 = **372 groups**
— Ignores the time of day completely, so a 3am reading gets compared to a 3pm reading in the same month. Not useful for hourly data 

**This is a core ML Principle:**
**More granularity is not always better. You need enough data in each group for statistics to be meaningful.**


In [9]:
df['Month']= df['date'].dt.month

In [10]:
df['Hour']= df['date'].dt.hour
df["dayofyear"] = df["date"].dt.dayofyear

In [11]:
df.groupby(['Month', 'Hour'])['temperature_2m'].agg(["mean", "std"]).size   
# this aproach gives us the most balanced granuity,helps us isolate sesonal patterens

576

In [12]:
df.groupby(['Month', 'dayofyear'])['temperature_2m'].agg(["mean", "std"]).size

752

In [13]:
df.groupby(['Month', 'Hour','dayofyear'])['temperature_2m'].agg(["mean", "std"]).size

18048

In [14]:
df.groupby([ 'Hour','dayofyear'])['temperature_2m'].agg(["mean", "std"]).size

17568

# Now that we have decided what we should group by lets go further and start calculation the Mean  and std for all the weather parameters.

In [15]:
#temp
df["expected_mean_temp"] = df.groupby(["Month", "Hour"])["temperature_2m"].transform("mean")
df["expected_std_temp"]  = df.groupby(["Month", "Hour"])["temperature_2m"].transform("std")
#relative_humidity_2m
df["expected_mean_humidity"] = df.groupby(["Month", "Hour"])["relative_humidity_2m"].transform("mean")
df["expected_std_humidity"]  = df.groupby(["Month", "Hour"])["relative_humidity_2m"].transform("std")
#precipitation
df["expected_mean_precipitation"] = df.groupby(["Month", "Hour"])["precipitation"].transform("mean")
df["expected_std_precipitation"]  = df.groupby(["Month", "Hour"])["precipitation"].transform("std")
#wind_speed_10m
df["expected_mean_wind_speed"] = df.groupby(["Month", "Hour"])["wind_speed_10m"].transform("mean")
df["expected_std_wind_speed"]  = df.groupby(["Month", "Hour"])["wind_speed_10m"].transform("std")
#apparent_temperature	
df["expected_mean_apparent_temperature"] = df.groupby(["Month", "Hour"])["apparent_temperature"].transform("mean")
df["expected_std_apparent_temperature"]  = df.groupby(["Month", "Hour"])["apparent_temperature"].transform("std")

In [16]:
df[(df['Month'] == 1) & (df['Hour'] == 0)] # Perfect! we can see that the mean and std was succesfully computed

,date,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,apparent_temperature,Month,Hour,dayofyear,expected_mean_temp,expected_std_temp,expected_mean_humidity,expected_std_humidity,expected_mean_precipitation,expected_std_precipitation,expected_mean_wind_speed,expected_std_wind_speed,expected_mean_apparent_temperature,expected_std_apparent_temperature
1,2020-01-01 00:00:00+01:00,0.20,88.342773,0.0,19.602652,-5.004395,1,0,1,-0.079032,3.846964,87.332932,9.74012,0.048387,0.192473,9.887184,5.607718,-3.859355,4.182662
25,2020-01-02 00:00:00+01:00,-1.50,90.512939,0.0,7.412853,-5.107805,1,0,2,-0.079032,3.846964,87.332932,9.74012,0.048387,0.192473,9.887184,5.607718,-3.859355,4.182662
49,2020-01-03 00:00:00+01:00,-3.95,97.407623,0.0,5.351785,-7.434804,1,0,3,-0.079032,3.846964,87.332932,9.74012,0.048387,0.192473,9.887184,5.607718,-3.859355,4.182662
73,2020-01-04 00:00:00+01:00,-2.75,97.796219,0.0,6.915374,-6.311033,1,0,4,-0.079032,3.846964,87.332932,9.74012,0.048387,0.192473,9.887184,5.607718,-3.859355,4.182662
97,2020-01-05 00:00:00+01:00,1.95,88.499329,0.7,26.886963,-4.057133,1,0,5,-0.079032,3.846964,87.332932,9.74012,0.048387,0.192473,9.887184,5.607718,-3.859355,4.182662
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53233,2026-01-27 00:00:00+01:00,1.45,97.169678,0.0,11.091564,-2.125223,1,0,27,-0.079032,3.846964,87.332932,9.74012,0.048387,0.192473,9.887184,5.607718,-3.859355,4.182662
53257,2026-01-28 00:00:00+01:00,-0.25,97.485100,0.0,6.120000,-3.358708,1,0,28,-0.079032,3.846964,87.332932,9.74012,0.048387,0.192473,9.887184,5.607718,-3.859355,4.182662
53281,2026-01-29 00:00:00+01:00,2.40,97.538490,0.1,10.583326,-0.930697,1,0,29,-0.079032,3.846964,87.332932,9.74012,0.048387,0.192473,9.887184,5.607718,-3.859355,4.182662
53305,2026-01-30 00:00:00+01:00,2.60,96.162270,0.3,9.743305,-0.608145,1,0,30,-0.079032,3.846964,87.332932,9.74012,0.048387,0.192473,9.887184,5.607718,-3.859355,4.182662


In [17]:
df[(df['Month'] == 1) & (df['Hour'] == 0)]['temperature_2m'].agg(['mean', 'std', 'count'])

mean      -0.079032
std        3.846964
count    217.000000
Name: temperature_2m, dtype: float64

In [18]:
df[["temperature_2m", "expected_mean_temp", "expected_std_temp",
    "apparent_temperature", "expected_mean_apparent_temperature", "expected_std_apparent_temperature"]].head(10)

,temperature_2m,expected_mean_temp,expected_std_temp,apparent_temperature,expected_mean_apparent_temperature,expected_std_apparent_temperature
0,0.25,1.640374,3.009946,-4.826773,-1.676121,3.503007
1,0.20,-0.079032,3.846964,-5.004395,-3.859355,4.182662
2,0.75,-0.018433,3.837927,-4.439463,-3.754334,4.179548
3,0.70,-0.095853,3.977882,-4.379635,-3.849583,4.258438
4,0.35,-0.229263,4.013070,-4.580226,-4.000408,4.291775
5,0.05,-0.298618,3.970032,-4.803813,-4.100819,4.275558
6,-0.30,-0.358525,3.958060,-4.854145,-4.188205,4.275017
7,-0.50,-0.399078,3.920264,-5.153129,-4.228047,4.248521
8,-0.60,-0.450691,3.996823,-5.303684,-4.282994,4.285009
9,-0.50,-0.227419,3.882710,-5.152630,-4.083187,4.119673


## Anomaly Scoring — Z-Score Method

To measure *how unusual* each observation is, we use the **Z-score formula**:

$$Z = \frac{X - \mu}{\sigma}$$

Where:
- $X$ = the actual observed value for that row
- $\mu$ = the expected mean for that Month + Hour group
- $\sigma$ = the expected standard deviation for that Month + Hour group

The Z-score tells us **how many standard deviations** the actual value is away from what is historically normal for that exact time of year and time of day.

### Interpreting the Z-score

| Z-score | Meaning |
|---|---|
| 0 | Exactly as expected |
| ±1 | Slightly unusual — happens ~32% of the time |
| ±2 | Moderately unusual — happens ~5% of the time |
| ±3 | Strongly anomalous — happens ~0.3% of the time |
| > ±3 | **Flagged as anomaly** |

### Why Z-score and not a fixed threshold?

A fixed threshold (e.g. "flag if temperature > 35°C") fails because it ignores context.  
35°C in August is normal. 35°C in January is impossible.  
The Z-score is **context-aware** — it compares each value against its own Month + Hour group, so the threshold adapts automatically to season and time of day.

### Applied to Each Variable

We compute one anomaly score per variable:

- **anomaly_score_temp** — how unusual is the temperature for this month and hour?
- **anomaly_score_humidity** — how unusual is the humidity for this month and hour?
- **anomaly_score_precipitation** — how unusual is the rainfall for this month and hour?
- **anomaly_score_wind_speed** — how unusual is the wind speed for this month and hour?
- **anomaly_score_apparent_temperature** — how unusual is the feels-like temperature for this month and hour?

A row is flagged as `is_anomaly = True` if **any** of its five Z-scores exceeds ±3.

In [19]:
df["anomaly_score_temp"] = (df["temperature_2m"] - df["expected_mean_temp"]) / df["expected_std_temp"]

df["anomaly_score_humidity"] = (df["relative_humidity_2m"] - df["expected_mean_humidity"]) / df["expected_std_humidity"]

df["anomaly_score_precipitation"] = (df["precipitation"] - df["expected_mean_precipitation"]) / df["expected_std_precipitation"]

df["anomaly_score_wind_speed"] = (df["wind_speed_10m"] - df["expected_mean_wind_speed"]) / df["expected_std_wind_speed"]

df["anomaly_score_apparent_temperature"] = (df["apparent_temperature"] - df["expected_mean_apparent_temperature"]) / df["expected_std_apparent_temperature"]

In [20]:
df[["anomaly_score_temp", "anomaly_score_humidity"]].head()

,anomaly_score_temp,anomaly_score_humidity
0,-0.461927,-0.454316
1,0.072533,0.103679
2,0.200221,-0.287150
3,0.200069,-0.151073
4,0.144344,0.049325


In [21]:
df.head()

,date,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,apparent_temperature,Month,Hour,dayofyear,expected_mean_temp,...,expected_std_precipitation,expected_mean_wind_speed,expected_std_wind_speed,expected_mean_apparent_temperature,expected_std_apparent_temperature,anomaly_score_temp,anomaly_score_humidity,anomaly_score_precipitation,anomaly_score_wind_speed,anomaly_score_apparent_temperature
0,2019-12-31 23:00:00+01:00,0.25,87.058907,0.0,18.584509,-4.826773,12,23,365,1.640374,...,0.285921,8.773926,5.727087,-1.676121,3.503007,-0.461927,-0.454316,-0.271194,1.713014,-0.899414
1,2020-01-01 00:00:00+01:00,0.20,88.342773,0.0,19.602652,-5.004395,1,0,1,-0.079032,...,0.192473,9.887184,5.607718,-3.859355,4.182662,0.072533,0.103679,-0.251396,1.732517,-0.273758
2,2020-01-01 01:00:00+01:00,0.75,84.590553,0.0,19.453327,-4.439463,1,1,1,-0.018433,...,0.215317,9.635202,5.750728,-3.754334,4.179548,0.200221,-0.287150,-0.203322,1.707284,-0.163924
3,2020-01-01 02:00:00+01:00,0.70,85.835953,0.0,18.844202,-4.379635,1,2,1,-0.095853,...,0.199094,9.671770,6.073979,-3.849583,4.258438,0.200069,-0.151073,-0.259238,1.510119,-0.124471
4,2020-01-01 03:00:00+01:00,0.35,88.032867,0.0,17.819090,-4.580226,1,3,1,-0.229263,...,0.224239,9.689234,5.922327,-4.000408,4.291775,0.144344,0.049325,-0.267160,1.372747,-0.135100


# Next Step — lets inspect When Anomalies Happen

This tells — are anomalies clustered in specific months (winter storms? summer thunderstorms?)? 
looks like a Very clean distribution! Anomalies spread across all 12 months — no single month dominates wildly.

In [22]:
anomaly_cols = [
    "anomaly_score_temp",
    "anomaly_score_humidity",
    "anomaly_score_precipitation",
    "anomaly_score_wind_speed",
    "anomaly_score_apparent_temperature"
]

df["is_anomaly"] = (df[anomaly_cols].abs() > 3).any(axis=1)

In [23]:
print(df["is_anomaly"].value_counts())
print(f"Anomaly rate: {df['is_anomaly'].mean():.2%}")

is_anomaly
False    53260
True      1964
Name: count, dtype: int64
Anomaly rate: 3.56%


In [24]:
for col in anomaly_cols:
    count = (df[col].abs() > 3).sum()
    print(f"{col}: {count}")

anomaly_score_temp: 47
anomaly_score_humidity: 194
anomaly_score_precipitation: 1205
anomaly_score_wind_speed: 546
anomaly_score_apparent_temperature: 37


In [25]:
anomalies = df[df["is_anomaly"] == True]
print(anomalies["Month"].value_counts().sort_index())

Month
1     195
2     174
3     157
4     125
5     136
6     128
7     125
8     167
9     175
10    191
11    203
12    188
Name: count, dtype: int64


## Monthly Anomaly Pattern

| Season | Months | Count | Pattern |
|---|---|---|---|
| Winter | Dec, Jan, Feb | 188, 195, 174 | Highest — storms, freezing rain |
| Autumn | Sep, Oct, Nov | 175, 191, 203 | High — wind storms, heavy rain |
| Summer | Jun, Jul, Aug | 128, 125, 167 | Lower — more stable |
| Spring | Mar, Apr, May | 157, 125, 136 | Lowest — mild transitions |

# Conclusion

## What Did We Find?

After analysing **55,224 hourly weather observations** from Budapest (2020–2026), 
the Z-score anomaly detection pipeline flagged **1,964 anomalous hours — a 3.56% anomaly rate.**

### Key Findings

- **Precipitation dominates anomalies** — 61.4% of all anomalies were driven by unusual rainfall or snowfall, 
  confirming that precipitation is the most unpredictable weather variable in Budapest
- **Wind speed is the second largest driver** — responsible for 27.8% of anomalies, 
  linked to sudden gusts and storm events
- **Temperature is highly predictable** — only 2.4% of anomalies came from temperature, 
  meaning Budapest's seasonal temperature pattern is very stable and consistent year over year
- **November is the most anomalous month** (203 anomalies), followed closely by October (191) and January (195), 
  reflecting Central European autumn storms and winter weather extremes
- **Summer (June–July) is the most stable period** — fewest anomalies despite occasional thunderstorms, 
  because those are already captured in the historical standard deviation

### Anomaly Rate by Variable

| Variable | Anomaly Count | Share of Total |
|---|---|---|
| Precipitation | 1,205 | 61.4% |
| Wind Speed | 546 | 27.8% |
| Humidity | 194 | 9.9% |
| Temperature | 47 | 2.4% |
| Apparent Temperature | 37 | 1.9% |

### Monthly Distribution

| Season | Months | Anomaly Count |
|---|---|---|
| Winter | Dec, Jan, Feb | 188, 195, 174 |
| Autumn | Sep, Oct, Nov | 175, 191, 203 |
| Summer | Jun, Jul, Aug | 128, 125, 167 |
| Spring | Mar, Apr, May | 157, 125, 136 |

In [ ]:
df.to_csv("weather_anomalies.csv", index=False)
print("Saved!", df.shape)